# 06 — Agentic RAG with LangGraph

Everything so far has been **traditional RAG**: one straight line, executed once.

```
query ──▶ embed ──▶ retrieve ──▶ stuff into prompt ──▶ answer
```

It has a structural blind spot: **it cannot tell when retrieval failed.** If the
vector store returns three irrelevant chunks, they go into the prompt anyway and the
model produces a confident answer built on nothing. You saw this measured in
notebook 05 — groundedness exists precisely because this happens.

**Agentic RAG** turns the line into a **graph with decisions in it**:

```
                     question
                        │
                        ▼
                    ┌────────┐   off-topic    ┌──────────────┐
                    │ route  │──────────────▶ │ answer direct│──▶ END
                    └────────┘                └──────────────┘
                        │ on-topic
                        ▼
                   ┌──────────┐
              ┌───▶│ retrieve │
              │    └──────────┘
              │         │
              │         ▼
              │  ┌───────────────┐  relevant   ┌──────────┐
              │  │ grade documents├───────────▶ │ generate │──▶ END
              │  └───────────────┘             └──────────┘
              │         │ irrelevant
              │         ▼
              │  ┌──────────────────┐  budget left
              └──┤ rewrite question │
                 └──────────────────┘  budget spent
                          │
                          ▼
                    ┌─────────┐
                    │ give up │──▶ END
                    └─────────┘
```

Three capabilities traditional RAG simply does not have:

1. **Routing** — "what is 17 × 4" never touches the index.
2. **Self-grading** — the pipeline can *detect* that retrieval failed.
3. **Query rewriting** — a failure is retried with better wording instead of
   producing a wrong answer.

And critically, a fourth outcome becomes reachable: **admitting it doesn't know.**

Ported from the repo's `agenticrag/1-agenticrag.ipynb` (OpenAI) to Groq + local
embeddings.

## 0. Setup

LangGraph is a state machine library. You define a shared `State`, write nodes as
plain functions that return updates to it, and connect them with edges — some
fixed, some conditional.

In [1]:
import sys
sys.path.append("..")

from src.data_loader import load_all_documents
from src.vector_store import FaissVectorStore
from src.agentic_rag import AgenticRAG, AgentState

store = FaissVectorStore(persist_directory="../faiss_store_nb06")
if store.exists():
    store.load()
else:
    store.build_from_documents(load_all_documents("../data"))

Loading embedding model for vector store: all-MiniLM-L6-v2 ...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Found 3 PDF files
  loaded attention_is_all_you_need.pdf: 3 pages
  loaded embedding_models_report.pdf: 3 pages
  loaded rag_systems_overview.pdf: 2 pages
Found 2 text files
  loaded machine_learning.txt
  loaded python_intro.txt
Found 1 CSV files
  loaded ai_models.csv: 4 rows
Found 1 JSON files
  loaded rag_faq.json: 4 entries
Found 1 Excel files
  loaded vector_databases.xlsx: 4 rows
Found 1 SQLite databases
  loaded llm_catalog.db: 4 rows from 1 table(s)
Found 1 Word files
  loaded retrieval_evaluation.docx
Total documents loaded: 27
Loading embedding model: all-MiniLM-L6-v2 ...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded. Embedding dimension: 384
Split 27 documents into 27 chunks


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape (27, 384)
Vector store now holds 27 vectors
Saved FAISS index + metadata to '../faiss_store_nb06'


## 1. The state

Every node reads from and writes to one shared dict. LangGraph merges whatever a
node returns into the state, so a node only returns the keys it changed.

Two fields carry the interesting design decisions:

- **`original_question` vs `question`** — `question` gets rewritten during retries,
  but grading and the final answer must use what the user *actually asked*.
- **`rewrites`** — the loop guard. Without it, a question the corpus can't answer
  loops forever. This is the single most important field in an agentic system.

In [2]:
print(AgentState.__doc__)

Shared state passed along the graph's edges.

LangGraph merges each node's returned dict into this state, so a node
only needs to return the keys it changed.

Keys
----
question:
    The current (possibly rewritten) question driving retrieval.
original_question:
    The user's untouched question, kept so the final answer addresses
    what was actually asked rather than a rewrite.
documents:
    Retrieved chunks, each ``{"content", "metadata", "distance"}``.
answer:
    The generated answer.
route:
    ``"retrieve"`` or ``"direct"`` — the router's decision.
grade:
    ``"relevant"`` or ``"irrelevant"`` — the document grader's verdict.
rewrites:
    How many times the question has been rewritten (loop guard).
trace:
    Human-readable log of the path taken — the point of the whole
    exercise is being able to see the decisions.



## 2. The nodes

Each is an ordinary function. Three of them make LLM calls to *decide* something,
which is what makes this "agentic" — control flow is determined at runtime by a
model, not hardcoded.

| Node | Does | LLM? |
|---|---|---|
| `route_question` | on-topic → retrieve, else answer directly | yes |
| `retrieve` | vector search | no |
| `grade_documents` | are these chunks relevant? | yes |
| `rewrite_question` | rephrase for better retrieval | yes |
| `generate` | answer from context | yes |
| `answer_directly` | answer without retrieval | yes |
| `give_up` | say the corpus can't answer it | no |

In [3]:
agent = AgenticRAG(store, top_k=3, max_rewrites=2)

Agentic RAG LLM initialized: openai/gpt-oss-120b


### The router's prompt is subtler than it looks

The obvious prompt — *"can you answer this without searching?"* — **breaks RAG.**
Ask that about "what is the attention mechanism" and any capable model says "yes, I
know that", skips retrieval, and answers from its training data. Your documents are
never consulted.

That's a real bug I hit building this: every question routed to `direct`, including
ones squarely in the corpus.

The fix is to ask about **topic membership**, not model capability:

> *"Reply RETRIEVE if the question is about any of those topics, **even if you
> already know the answer**."*

The default also leans toward retrieving, because a needless retrieval is cheap
while a skipped one loses the grounding entirely.

In [4]:
print(agent.route_question.__doc__)

Decide whether this question needs the knowledge base at all.

Saves a pointless retrieval for questions the model can answer from
its own weights ("what is 2 + 2"), and keeps unrelated questions from
dragging in irrelevant context that would only confuse the answer.



## 3. The graph

Nodes plus edges. Note the two `add_conditional_edges` calls — those are the
branches — and `add_edge("rewrite_question", "retrieve")`, which creates the
**loop** back into retrieval.

In [5]:
print(agent._build_graph.__doc__)
print("\nCompiled graph nodes:")
for node in agent.graph.get_graph().nodes:
    print("  •", node)

Wire the nodes and edges into a compiled LangGraph state machine.

Compiled graph nodes:
  • __start__
  • route_question
  • retrieve
  • grade_documents
  • rewrite_question
  • generate
  • answer_directly
  • give_up
  • __end__


In [6]:
# LangGraph can render the structure as mermaid
print(agent.graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	route_question(route_question)
	retrieve(retrieve)
	grade_documents(grade_documents)
	rewrite_question(rewrite_question)
	generate(generate)
	answer_directly(answer_directly)
	give_up(give_up)
	__end__([<p>__end__</p>]):::last
	__start__ --> route_question;
	grade_documents -.-> generate;
	grade_documents -.-> give_up;
	grade_documents -.-> rewrite_question;
	retrieve --> grade_documents;
	rewrite_question --> retrieve;
	route_question -.-> answer_directly;
	route_question -.-> retrieve;
	answer_directly --> __end__;
	generate --> __end__;
	give_up --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 4. Three cases, three different paths

The point of the trace output is that you can *see* the decisions.

### Case 1 — a question the corpus answers well

In [7]:
result = agent.print_run("What is the attention mechanism?")

Q: What is the attention mechanism?
   -> route -> retrieve
   -> retrieve -> 3 chunks from ['attention_is_all_you_need.pdf']
   -> grade -> relevant
   -> generate
   sources: ['attention_is_all_you_need.pdf']
A: The attention mechanism maps a **query** vector and a set of **key‑value** pairs to an output vector by computing a compatibility score between the query and each key, turning these scores into weights (e.g., via softmax), and then taking a weighted sum of the corresponding values. This lets the model focus on the most relevant parts of the input when producing each representation.



`route → retrieve`, `grade → relevant`, `generate`. The straight path — same
result traditional RAG would give, reached with one extra grading call.

### Case 2 — clearly off-topic

In [8]:
result = agent.print_run("What is 17 multiplied by 4?")

Q: What is 17 multiplied by 4?
   -> route -> direct
   -> answer_directly
A: 68



`route → direct`. No retrieval, no wasted vector search, and — importantly — no
irrelevant context dragged into the prompt to confuse the answer.

### Case 3 — on-topic but genuinely absent from the corpus

This is the case that justifies the whole architecture. HNSW indexing is squarely
on-topic (vector databases), so routing sends it to retrieval — but nothing in our
corpus explains it.

In [9]:
result = agent.print_run("How does HNSW graph indexing work internally?")

Q: How does HNSW graph indexing work internally?
   -> route -> retrieve
   -> retrieve -> 3 chunks from ['attention_is_all_you_need.pdf', 'rag_faq.json']
   -> grade -> irrelevant
   -> rewrite -> 'What is the internal mechanism of Hierarchical Navigable Small World (HNSW) graph indexing?'
   -> retrieve -> 3 chunks from ['attention_is_all_you_need.pdf', 'rag_faq.json', 'vector_databases.xlsx']
   -> grade -> irrelevant
   -> rewrite -> 'How does the Hierarchical Navigable Small World (HNSW) algorithm construct and traverse its multi‑layer graph index?'
   -> retrieve -> 3 chunks from ['attention_is_all_you_need.pdf', 'llm_catalog.db', 'rag_systems_overview.pdf']
   -> grade -> irrelevant
   -> give_up (rewrite budget exhausted)
   sources: ['attention_is_all_you_need.pdf', 'llm_catalog.db', 'rag_systems_overview.pdf']
A: I could not find relevant information in the knowledge base to answer this question.



Follow that trace:

1. Retrieved chunks from `attention_is_all_you_need.pdf` and `rag_faq.json`.
2. **Graded them irrelevant** — correctly.
3. Rewrote the question with more precise terminology, retried.
4. Still irrelevant. Rewrote again, retried.
5. Budget exhausted → **gave up honestly.**

Compare against what traditional RAG does with the same question: it takes those
same irrelevant attention/FAQ chunks and writes a confident, plausible answer about
HNSW that is entirely unsupported. Let's verify that claim rather than assert it.

In [10]:
from src.search import RAGSearch

traditional = RAGSearch(store)
plain = traditional.search_and_summarize("How does HNSW graph indexing work internally?", top_k=3)
print("TRADITIONAL RAG on the same question:")
print("  sources used:", sorted({s["source_file"] for s in plain["sources"]}))
print("  answer:", plain["answer"][:420])

Groq LLM initialized: openai/gpt-oss-120b


TRADITIONAL RAG on the same question:
  sources used: ['attention_is_all_you_need.pdf', 'rag_faq.json']
  answer: **HNSW (Hierarchical Navigable Small World) graph indexing** builds a multi‑level proximity graph that lets you perform fast approximate nearest‑neighbor (ANN) searches.

1. **Structure – layered graph**  
   * Each data point is inserted into one or more layers.  
   * The top‑most layer is very sparse (few nodes, long‑range links); lower layers become progressively denser, with the bottom layer containing every poi


Two systems, same corpus, same question, opposite behaviour. The agentic one
says "I don't know"; the traditional one answers anyway from whatever came back.

Which you want depends on the stakes — but only one of them gives you the choice.

## 5. What this costs

Nothing is free:

| | Traditional RAG | Agentic RAG |
|---|---|---|
| LLM calls (happy path) | 1 | 3 (route + grade + generate) |
| LLM calls (worst case) | 1 | 8 (with 2 rewrite loops) |
| Latency | one round trip | several, serially |
| Failure mode | confident wrong answer | honest "I don't know" |

So agentic RAG is **~3× the cost on the happy path** and slower. Use it where a
wrong answer is expensive — support, compliance, medical, legal — and traditional
RAG where speed matters and users can judge the output themselves.

## 6. Where to take it

The graph is the extension point. Natural next nodes:

- **`web_search`** — on `give_up`, fall back to an external search instead of
  surrendering. This is what makes systems like Perplexity work.
- **`check_groundedness`** — reuse notebook 05's judge *inside* the graph, and loop
  back to `generate` if the answer isn't supported. Self-correcting generation.
- **`decompose_question`** — split multi-hop questions into sub-questions,
  retrieve for each, then synthesise.
- **Multiple retrievers** — route to different indexes by topic.

That progression, from a straight line to a graph that grades and corrects itself,
is the whole difference between a RAG demo and a RAG system.